In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Ejecutar trabajos en una sesión

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
scipy~=1.16.3
```
</details>
> **Note:** Los usuarios del Plan Abierto no pueden enviar trabajos en sesión. Los workloads deben ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).

Usa sesiones cuando necesites acceso dedicado y exclusivo al QPU.

## Configuración para usar sesiones
Antes de iniciar una sesión, debes [configurar Qiskit Runtime](./install-qiskit) e inicializarlo como servicio:

In [1]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Session,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

service = QiskitRuntimeService()

## Abrir una sesión
Puedes abrir una sesión de runtime usando el gestor de contexto `with Session(...)` o inicializando la clase `Session`.
Al iniciar una sesión, debes especificar un QPU pasando un objeto `backend`. La sesión comienza cuando su primer trabajo empieza a ejecutarse.

> **Note:** Si abres una sesión pero no envías ningún trabajo durante 30 minutos, la sesión se cierra automáticamente.

**Clase Session**

> **Caution:** El siguiente bloque de código devolverá un error para los usuarios del Plan Abierto porque usa sesiones. Los workloads del Plan Abierto solo pueden ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).

In [2]:
backend = service.least_busy(operational=True, simulator=False)
session = Session(backend=backend)
estimator = Estimator(mode=session)
sampler = Sampler(mode=session)
# Close the session because no context manager was used.
session.close()

**Gestor de contexto**

El gestor de contexto abre y cierra la sesión automáticamente.

> **Caution:** El siguiente bloque de código devolverá un error para los usuarios del Plan Abierto porque usa sesiones. Los workloads del Plan Abierto solo pueden ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).

In [3]:
from qiskit_ibm_runtime import (
    Session,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

backend = service.least_busy(operational=True, simulator=False)
with Session(backend=backend):
    estimator = Estimator()
    sampler = Sampler()

<span id="specify-length"></span>
## Duración de la sesión
El tiempo de vida máximo (TTL) de la sesión determina cuánto tiempo puede ejecutarse una sesión. Puedes establecer este valor con el parámetro `max_time`. Este valor debe superar el tiempo de ejecución del trabajo más largo.

Este temporizador comienza cuando la sesión se inicia. Al alcanzar el valor, la sesión se cierra. Los trabajos que estén en ejecución terminarán, pero los trabajos aún en cola fallarán.

> **Caution:** El siguiente bloque de código devolverá un error para los usuarios del Plan Abierto porque usa sesiones. Los workloads del Plan Abierto solo pueden ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).

In [4]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

# This cell is hidden from users
service = QiskitRuntimeService()
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
transpiled_circuit_sampler = transpiled_circuit
transpiled_circuit_sampler.measure_all()

# Create parameters and mapped observables to submit
params = np.random.uniform(size=(2, 3)).T
observables = [
    SparsePauliOp(["XX", "IY"], [0.5, 0.5]),
    SparsePauliOp("XX"),
    SparsePauliOp("IY"),
]
mapped_observables = [
    [observable.apply_layout(transpiled_circuit.layout)]
    for observable in observables
]

sampler_pub = (transpiled_circuit_sampler, params)
estimator_pub = (transpiled_circuit_sampler, mapped_observables, params)

In [5]:
with Session(backend=backend) as session:
    estimator = Estimator()
    sampler = Sampler()
    job1 = estimator.run([estimator_pub])
    job2 = sampler.run([sampler_pub])

# The session is no longer accepting jobs but the submitted job will run to completion.
result = job1.result()
result2 = job2.result()

<Admonition type="tip">
If you are not using a context manager, manually close the session to avoid unwanted cost. You can close a session as soon as you are done submitting jobs to it. When a session is closed with `session.close()`, it no longer accepts new jobs, but the already submitted jobs will still run until completion and their results can be retrieved.
</Admonition>


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [6]:
session = Session(backend=backend)

# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `session=`
estimator = Estimator(mode=session)
sampler = Sampler(mode=session)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])
print(f"Result1: {job1.result()}")
print(f"Result2: {job2.result()}")

# Manually close the session. Running and queued jobs will run to completion.
session.close()

Result1: PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 2), dtype=float64>), stds=np.ndarray(<shape=(3, 2), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 2), dtype=float64>), shape=(3, 2)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


Result2: PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(3, 2), num_shots=4096, num_bits=2>), meas0=BitArray(<shape=(3, 2), num_shots=4096, num_bits=133>), shape=(3, 2)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 07:53:15', stop='2026-01-15 07:53:21', size=24576>)])}, 'version': 2})


> **Tip:** Si no estás usando un gestor de contexto, cierra la sesión manualmente para evitar costos no deseados. Puedes cerrar una sesión en cuanto termines de enviar trabajos. Cuando una sesión se cierra con `session.close()`, ya no acepta nuevos trabajos, pero los trabajos ya enviados seguirán ejecutándose hasta completarse y sus resultados podrán recuperarse.

> **Caution:** El siguiente bloque de código devolverá un error para los usuarios del Plan Abierto porque usa sesiones. Los workloads del Plan Abierto solo pueden ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).

In [7]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Session,
    EstimatorV2 as Estimator,
)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

with Session(backend=backend) as session:
    print(session.details())

{'id': 'be84569d-86b5-4a7f-be5e-7d33e80dc220', 'backend_name': 'ibm_torino', 'interactive_timeout': 60, 'max_time': 28800, 'active_timeout': 28800, 'state': 'open', 'accepting_jobs': True, 'last_job_started': None, 'last_job_completed': None, 'started_at': None, 'closed_at': None, 'activated_at': None, 'mode': 'dedicated', 'usage_time': None}


## Usage patterns

Sessions are especially useful for algorithms that require frequent communication between classical and quantum resources.

Example: Run an iterative workload that uses the classical SciPy optimizer to minimize a cost function. In this model, SciPy uses the output of the cost function to calculate its next input.


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [8]:
from scipy.optimize import minimize
from qiskit.circuit.library import efficient_su2


def cost_func(params, ansatz, hamiltonian, estimator):
    # Return estimate of energy from estimator

    energy = sum(
        estimator.run([(ansatz, hamiltonian, params)]).result()[0].data.evs
    )
    return energy


hamiltonian = SparsePauliOp.from_list(
    [("YZ", 0.3980), ("ZI", -0.3980), ("ZZ", -0.0113), ("XX", 0.1810)]
)
su2_ansatz = efficient_su2(hamiltonian.num_qubits)
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
ansatz = pm.run(su2_ansatz)
mapped_hamiltonian = [
    operator.apply_layout(ansatz.layout) for operator in hamiltonian
]

num_params = ansatz.num_parameters
x0 = 2 * np.pi * np.random.random(num_params)

session = Session(backend=backend)


# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `session=`
estimator = Estimator(mode=session, options={"default_shots": int(1e4)})
res = minimize(
    cost_func,
    x0,
    args=(ansatz, mapped_hamiltonian, estimator),
    method="cobyla",
    options={"maxiter": 25},
)

# Close the session because no context manager was used.
session.close()

<span id="two-vqe"></span>
## Run two VQE algorithms in a session by using threading

You can get more out of a session by running multiple workloads simultaneously. The following example shows how you can run two VQE algorithms, each using a different classical optimizer, simultaneously inside a single session. Job tags are also used to differentiate jobs from each workload.


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [9]:
from concurrent.futures import ThreadPoolExecutor
from qiskit_ibm_runtime import EstimatorV2 as Estimator


def minimize_thread(estimator, method):
    return minimize(
        cost_func,
        x0,
        args=(ansatz, mapped_hamiltonian, estimator),
        method=method,
        options={"maxiter": 25},
    )


with Session(backend=backend), ThreadPoolExecutor() as executor:
    estimator1 = Estimator()
    estimator2 = Estimator()

    # Use different tags to differentiate the jobs.
    estimator1.options.environment.job_tags = ["cobyla"]
    estimator2.options.environment.job_tags = ["nelder-mead"]

    # Submit the two workloads.
    cobyla_future = executor.submit(minimize_thread, estimator1, "cobyla")
    nelder_mead_future = executor.submit(
        minimize_thread, estimator2, "nelder-mead"
    )

    # Get workload results.
    cobyla_result = cobyla_future.result()
    nelder_mead_result = nelder_mead_future.result()

<span id="session-status"></span>
## Verificar el estado de la sesión
Puedes consultar el estado de una sesión para entender su estado actual usando `session.status()` o visitando la página de [Workloads](https://quantum.cloud.ibm.com/workloads).

El estado de la sesión puede ser uno de los siguientes:

- `Pending` (Pendiente): La sesión no ha comenzado o ha sido desactivada. El siguiente trabajo de la sesión debe esperar en la cola como cualquier otro trabajo.
- `In progress, accepting new jobs` (En progreso, aceptando nuevos trabajos): La sesión está activa y acepta nuevos trabajos.
- `In progress, not accepting new jobs` (En progreso, no acepta nuevos trabajos): La sesión está activa pero no acepta nuevos trabajos. El envío de trabajos a la sesión es rechazado, pero los trabajos pendientes se ejecutarán hasta completarse. La sesión se cierra automáticamente una vez que todos los trabajos terminen.
- `Closed` (Cerrada): Se alcanzó el valor máximo de tiempo de espera de la sesión o la sesión fue cerrada explícitamente.

<span id="session-details"></span>
## Determinar los detalles de la sesión
Para obtener una visión completa de la configuración y el estado de una sesión, usa el método `session.details()`.

> **Caution:** El siguiente bloque de código devolverá un error para los usuarios del Plan Abierto porque usa sesiones. Los workloads del Plan Abierto solo pueden ejecutarse en [modo de trabajo](/guides/execution-modes#job-mode) o [modo por lotes](/guides/execution-modes#batch-mode).